# 1. Data Exploration

**Purpose**: Understand data before training. Run once, refer back as needed.

## Goals
1. Load samples from all three tables
2. Field coverage comparison
3. ROR ID overlap analysis (ground truth availability)
4. Blocking rule coverage simulation
5. Name quality distribution
6. Export exploration summary


---
## Setup


In [ ]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26')

import pandas as pd
import numpy as np
from collections import Counter

from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    save_json,
    field_coverage_table,
    describe_dataframe
)
from data_prep import (
    load_dim_org,
    load_grid,
    load_mismatched,
    create_unified_schema,
    filter_bad_records,
    is_generic_name
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)

print("Imports loaded successfully")


In [ ]:
# Initialize database connection
db = DatabaseManager()
print("Database manager ready")


---
## 1. Load Sample Data


In [ ]:
# Configuration for exploration
SAMPLE_SIZE = config.sampling.EXPLORATION_SAMPLE_PER_SOURCE
print(f"Exploration sample size: {SAMPLE_SIZE} per source")


In [ ]:
# Load dim_organization sample
dim_org_raw = load_dim_org(db, sample_size=105000)
print(f"\ndim_organization: {len(dim_org_raw):,} rows")
dim_org_raw.head(3)


In [ ]:
# Load GRID sample
grid_raw = load_grid(db, sample_size=105000)
print(f"\nGRID: {len(grid_raw):,} rows")
grid_raw.head(3)


In [ ]:
grid_raw['ror_id'].isna().sum()

In [ ]:
# Load mismatched sample (stratified by source)
mismatched_raw = load_mismatched(db, samples_per_source=SAMPLE_SIZE)
print(f"\nMismatched: {len(mismatched_raw):,} rows")
print(f"\nSource tables:")
print(mismatched_raw['source_table'].value_counts())


---
## 2. Field Coverage Comparison


In [ ]:
# Key fields to analyze
KEY_FIELDS = [
    'name', 'name_clean', 'name_normalized', 
    'name_prefix_5', 'name_prefix_10',
    'org_type', 'country_code', 'city',
    'latitude', 'longitude',
    'ror_id', 'grid_id'
]


In [ ]:
# Create unified schemas for comparison
dim_org_unified = create_unified_schema(dim_org_raw, 'dim_org')
grid_unified = create_unified_schema(grid_raw, 'grid')
mismatched_unified = create_unified_schema(mismatched_raw, 'mismatched')


In [ ]:
# Field coverage for each table
print("FIELD COVERAGE COMPARISON")
print("=" * 70)

coverage_dim = field_coverage_table(dim_org_unified, KEY_FIELDS)
coverage_grid = field_coverage_table(grid_unified, KEY_FIELDS)
coverage_mis = field_coverage_table(mismatched_unified, KEY_FIELDS)

# Combine into single view
coverage_combined = pd.DataFrame({
    'field': KEY_FIELDS,
    'dim_org_%': [coverage_dim[coverage_dim['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS],
    'grid_%': [coverage_grid[coverage_grid['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS],
    'mismatched_%': [coverage_mis[coverage_mis['field']==f]['coverage_pct'].values[0] for f in KEY_FIELDS]
})

coverage_combined


In [ ]:
# Visual coverage comparison
print("\nVISUAL FIELD COVERAGE")
print("=" * 70)
print(f"{'Field':<20} | {'dim_org':<15} | {'grid':<15} | {'mismatched':<15}")
print("-" * 70)

for _, row in coverage_combined.iterrows():
    def bar(pct):
        filled = int(pct / 10)
        return '#' * filled + '.' * (10 - filled) + f" {pct:5.1f}%"
    
    print(f"{row['field']:<20} | {bar(row['dim_org_%']):<15} | {bar(row['grid_%']):<15} | {bar(row['mismatched_%']):<15}")


---
## 3. ROR ID Overlap Analysis (Ground Truth)


In [ ]:
# Count records with ROR IDs
dim_with_ror = dim_org_unified[dim_org_unified['ror_id'].notna()]
grid_with_ror = grid_unified[grid_unified['ror_id'].notna()]

print("ROR ID AVAILABILITY")
print("=" * 50)
print(f"dim_organization with ROR: {len(dim_with_ror):,} / {len(dim_org_unified):,} ({100*len(dim_with_ror)/len(dim_org_unified):.1f}%)")
print(f"GRID with ROR:             {len(grid_with_ror):,} / {len(grid_unified):,} ({100*len(grid_with_ror)/len(grid_unified):.1f}%)")


In [ ]:
# Find overlapping ROR IDs (these become ground truth pairs)
dim_ror_set = set(dim_with_ror['ror_id'].dropna())
grid_ror_set = set(grid_with_ror['ror_id'].dropna())

overlap_ror = dim_ror_set & grid_ror_set

print("\nROR ID OVERLAP (Ground Truth Potential)")
print("=" * 50)
print(f"Unique ROR IDs in dim_org: {len(dim_ror_set):,}")
print(f"Unique ROR IDs in GRID:    {len(grid_ror_set):,}")
print(f"Overlapping ROR IDs:       {len(overlap_ror):,}")
print(f"")
print(f"This gives us {len(overlap_ror):,} ground truth matches for training!")


In [ ]:
# Sample of matching pairs
if len(overlap_ror) > 0:
    print("\nSAMPLE GROUND TRUTH PAIRS")
    print("=" * 80)
    
    sample_rors = list(overlap_ror)[:5]
    for ror in sample_rors:
        dim_name = dim_with_ror[dim_with_ror['ror_id'] == ror]['name'].iloc[0]
        grid_name = grid_with_ror[grid_with_ror['ror_id'] == ror]['name'].iloc[0]
        print(f"\nROR: {ror}")
        print(f"  dim_org: {dim_name[:60]}")
        print(f"  GRID:    {grid_name[:60]}")


---
## 4. Blocking Rule Coverage Simulation


In [ ]:
# Analyze blocking key distributions
print("BLOCKING KEY ANALYSIS")
print("=" * 60)

def analyze_blocking_key(df, key_col, name):
    """Analyze a blocking key column"""
    non_null = df[key_col].notna().sum()
    coverage = 100 * non_null / len(df)
    unique_values = df[key_col].nunique()
    
    # Estimate pairs generated
    value_counts = df[key_col].value_counts()
    pairs_per_key = (value_counts * (value_counts - 1) / 2).sum()
    
    print(f"\n{name}:")
    print(f"  Coverage: {non_null:,} / {len(df):,} ({coverage:.1f}%)")
    print(f"  Unique values: {unique_values:,}")
    print(f"  Estimated pairs: {pairs_per_key:,.0f}")
    print(f"  Top values: {value_counts.head(5).to_dict()}")
    
    return {
        'key': name,
        'coverage_pct': coverage,
        'unique_values': unique_values,
        'estimated_pairs': pairs_per_key
    }

blocking_stats = []
for df, source in [(dim_org_unified, 'dim_org'), (grid_unified, 'grid'), (mismatched_unified, 'mismatched')]:
    print(f"\n{'='*60}")
    print(f"SOURCE: {source}")
    print(f"{'='*60}")
    
    for key in ['name_prefix_5', 'name_prefix_10', 'country_code']:
        if key in df.columns:
            stats = analyze_blocking_key(df, key, key)
            stats['source'] = source
            blocking_stats.append(stats)


In [ ]:
# Estimate cross-table blocking coverage
print("\nCROSS-TABLE BLOCKING SIMULATION")
print("=" * 60)

# How many mismatched records would be blocked with training data?
for key in ['name_prefix_5', 'name_prefix_10', 'country_code']:
    if key in mismatched_unified.columns and key in dim_org_unified.columns:
        mis_keys = set(mismatched_unified[key].dropna())
        dim_keys = set(dim_org_unified[key].dropna())
        
        overlap = len(mis_keys & dim_keys)
        mis_coverage = 100 * overlap / len(mis_keys) if mis_keys else 0
        
        print(f"\n{key}:")
        print(f"  Mismatched keys: {len(mis_keys):,}")
        print(f"  dim_org keys:    {len(dim_keys):,}")
        print(f"  Overlap:         {overlap:,} ({mis_coverage:.1f}% of mismatched)")


In [ ]:
# Identify records with NO blocking key
no_blocking_key = mismatched_unified[
    mismatched_unified['name_prefix_5'].isna() & 
    mismatched_unified['name_prefix_10'].isna() &
    mismatched_unified['country_code'].isna()
]

print(f"\nRECORDS WITH NO USABLE BLOCKING KEY")
print(f"=" * 50)
print(f"Count: {len(no_blocking_key):,} / {len(mismatched_unified):,} ({100*len(no_blocking_key)/len(mismatched_unified):.1f}%)")

if len(no_blocking_key) > 0:
    print(f"\nSamples:")
    display(no_blocking_key[['unique_id', 'name', 'source']].head(10))


---
## 5. Name Quality Distribution


In [ ]:
# Name length distribution
print("NAME LENGTH DISTRIBUTION")
print("=" * 50)

for df, source in [(dim_org_unified, 'dim_org'), (grid_unified, 'grid'), (mismatched_unified, 'mismatched')]:
    lengths = df['name'].str.len()
    print(f"\n{source}:")
    print(f"  Min: {lengths.min()}, Max: {lengths.max()}")
    print(f"  Mean: {lengths.mean():.1f}, Median: {lengths.median():.1f}")
    print(f"  Very short (<5): {(lengths < 5).sum():,}")
    print(f"  Very long (>100): {(lengths > 100).sum():,}")


In [ ]:
# Generic name detection in mismatched
print("\nGENERIC NAME DETECTION (mismatched only)")
print("=" * 50)

mismatched_unified['is_generic'] = mismatched_unified['name'].apply(is_generic_name)
generic_count = mismatched_unified['is_generic'].sum()
print(f"Generic names: {generic_count:,} / {len(mismatched_unified):,} ({100*generic_count/len(mismatched_unified):.1f}%)")

if generic_count > 0:
    print(f"\nExamples:")
    display(mismatched_unified[mismatched_unified['is_generic']][['name', 'source']].head(10))


In [ ]:
# Most common names (potential duplicates or term frequency issues)
print("\nMOST COMMON NAMES")
print("=" * 50)

for df, source in [(dim_org_unified, 'dim_org'), (mismatched_unified, 'mismatched')]:
    name_counts = df['name'].value_counts().head(10)
    print(f"\n{source}:")
    for name, count in name_counts.items():
        if count > 1:
            print(f"  {count:>4}x | {name[:60]}")


In [ ]:
# Organization category analysis (sub-orgs vs standard orgs)
print("\nORGANIZATION CATEGORIES")
print("=" * 50)

def classify_org(name):
    if pd.isna(name):
        return 'Unknown'
    name_lower = name.lower()
    if 'department' in name_lower:
        return 'Department'
    elif 'division' in name_lower:
        return 'Division'
    elif 'center' in name_lower or 'centre' in name_lower:
        return 'Center'
    elif 'institute' in name_lower:
        return 'Institute'
    elif 'school of' in name_lower:
        return 'School'
    elif 'university' in name_lower:
        return 'University'
    elif 'hospital' in name_lower:
        return 'Hospital'
    else:
        return 'Other'

mismatched_unified['org_category'] = mismatched_unified['name'].apply(classify_org)
category_counts = mismatched_unified['org_category'].value_counts()

for cat, count in category_counts.items():
    pct = 100 * count / len(mismatched_unified)
    print(f"  {cat:<15} | {count:>6,} ({pct:5.1f}%)")


---
## 6. Filter Impact Analysis


In [ ]:
# Test filtering on mismatched data
print("FILTER IMPACT ANALYSIS")
print("=" * 60)

filtered_df, removed_df = filter_bad_records(mismatched_unified)


In [ ]:
# Removal breakdown
if '_remove_reason' in removed_df.columns:
    print("\nRemoval reasons:")
    print(removed_df['_remove_reason'].value_counts())

# Sample removed records
if len(removed_df) > 0:
    print("\nSample removed records:")
    display(removed_df[['unique_id', 'name', '_remove_reason']].head(10))


---
## 7. Export Exploration Summary


In [ ]:
# Compile summary statistics
exploration_summary = {
    'timestamp': pd.Timestamp.now().isoformat(),
    
    'data_sizes': {
        'dim_org_sample': len(dim_org_unified),
        'grid_sample': len(grid_unified),
        'mismatched_sample': len(mismatched_unified)
    },
    
    'ror_overlap': {
        'dim_org_with_ror': len(dim_with_ror),
        'grid_with_ror': len(grid_with_ror),
        'overlapping_ror_ids': len(overlap_ror),
        'ground_truth_pairs_available': len(overlap_ror)
    },
    
    'field_coverage': coverage_combined.to_dict('records'),
    
    'blocking_stats': blocking_stats,
    
    'name_quality': {
        'generic_names_count': int(generic_count),
        'generic_names_pct': float(100 * generic_count / len(mismatched_unified)),
        'no_blocking_key_count': len(no_blocking_key),
        'no_blocking_key_pct': float(100 * len(no_blocking_key) / len(mismatched_unified))
    },
    
    'filtering': {
        'records_after_filter': len(filtered_df),
        'records_removed': len(removed_df),
        'removal_pct': float(100 * len(removed_df) / len(mismatched_unified))
    }
}

save_json(exploration_summary, config.paths.EXPLORATION_SUMMARY, "Exploration summary")


In [ ]:
# Print key findings
print("\n" + "=" * 70)
print("KEY FINDINGS SUMMARY")
print("=" * 70)

print(f"""
DATA SIZES:
  - dim_organization sample: {len(dim_org_unified):,} records
  - GRID sample: {len(grid_unified):,} records
  - Mismatched sample: {len(mismatched_unified):,} records

GROUND TRUTH (ROR Overlap):
  - dim_org with ROR: {100*len(dim_with_ror)/len(dim_org_unified):.1f}%
  - GRID with ROR: {100*len(grid_with_ror)/len(grid_unified):.1f}%
  - Overlapping ROR IDs: {len(overlap_ror):,} (available for training)

BLOCKING COVERAGE:
  - Records with no blocking key: {len(no_blocking_key):,} ({100*len(no_blocking_key)/len(mismatched_unified):.1f}%)

DATA QUALITY:
  - Generic names: {generic_count:,} ({100*generic_count/len(mismatched_unified):.1f}%)
  - Records removed by filters: {len(removed_df):,} ({100*len(removed_df)/len(mismatched_unified):.1f}%)
  - Records remaining for matching: {len(filtered_df):,}

RECOMMENDATIONS:
  1. Use ROR matches as ground truth for training
  2. Filter out generic names before matching
  3. Use multiple blocking rules to maximize coverage
  4. Consider feature-sparse training for inference robustness
""")


In [ ]:
# Cleanup
db.close()
print("\nExploration complete. Summary saved to:", config.paths.EXPLORATION_SUMMARY)
